# kNN-TN Parity Analysis: Python vs R

This notebook demonstrates that the Python implementation matches R's GSimp kNN-TN output to machine precision, and benchmarks performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import os
import sys

sys.path.insert(0, os.path.join("..", "src"))
from impute_knn_tn import impute_knn
from impute_knn_tn.knn_engine import _USE_CYTHON

REF_DIR = os.path.join("..", "tests", "reference")
print(
    f"Cython acceleration: {'enabled' if _USE_CYTHON else 'disabled (pure Python fallback)'}"
)

## 1. Load reference datasets

In [ ]:
def load_reference(name, distance):
    """Load input, R output, and run Python imputation."""
    config = f"{name}_{distance}"
    d = os.path.join(REF_DIR, config)
    inp = pd.read_csv(os.path.join(d, "input_matrix.csv"), index_col=0)
    out = pd.read_csv(os.path.join(d, "output_matrix.csv"), index_col=0)
    log2_file = os.path.join(d, "log2.txt")
    use_log2 = os.path.exists(log2_file) and open(log2_file).read().strip() == "true"
    r_timing_file = os.path.join(d, "timing.txt")
    r_time = (
        float(open(r_timing_file).read().strip())
        if os.path.exists(r_timing_file)
        else None
    )

    mat = inp.values.astype(np.float64)
    ref = out.values.astype(np.float64)

    if use_log2:
        work = np.log2(mat)
    else:
        work = mat.copy()

    # Warmup
    impute_knn(work.copy(), k=5, distance=distance, perc=0.01)

    # Timed run
    t0 = time.perf_counter()
    py_result = impute_knn(work, k=5, distance=distance, perc=0.01)
    py_time = time.perf_counter() - t0

    if use_log2:
        py_result = np.power(2.0, py_result)

    return {
        "name": name,
        "distance": distance,
        "input": mat,
        "r_output": ref,
        "py_output": py_result,
        "py_time": py_time,
        "r_time": r_time,
        "features": mat.shape[0],
        "samples": mat.shape[1],
        "n_missing": int(np.sum(np.isnan(mat))),
        "use_log2": use_log2,
    }

In [ ]:
DATASETS = [
    ("targeted", "truncation"),
    ("targeted", "correlation"),
    ("untargeted", "truncation"),
    ("untargeted", "correlation"),
    ("real_data", "truncation"),
    ("real_data", "correlation"),
    ("sim", "truncation"),
    ("sim", "correlation"),
    ("synthetic_1000", "truncation"),
    ("synthetic_1000", "correlation"),
]

results = []
for name, dist in DATASETS:
    config = f"{name}_{dist}"
    if os.path.isdir(os.path.join(REF_DIR, config)):
        r = load_reference(name, dist)
        results.append(r)
        print(
            f"{config:<30} features={r['features']:>5}  samples={r['samples']:>4}  NAs={r['n_missing']:>5}"
        )

print(f"\nLoaded {len(results)} datasets.")

## 2. Parity: Python vs R output correlation

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, r in enumerate(results):
    ax = axes[idx]
    py_vals = r["py_output"].ravel()
    r_vals = r["r_output"].ravel()

    # Plot in log2 space for datasets that use log2 transform
    if r["use_log2"]:
        py_flat = np.log2(np.clip(py_vals, 1e-300, None))
        r_flat = np.log2(np.clip(r_vals, 1e-300, None))
        scale_label = "log2"
    else:
        py_flat = py_vals
        r_flat = r_vals
        scale_label = "linear"

    ax.scatter(r_flat, py_flat, s=1, alpha=0.5)

    # Perfect agreement line
    lims = [min(r_flat.min(), py_flat.min()), max(r_flat.max(), py_flat.max())]
    ax.plot(lims, lims, "r--", linewidth=0.5, alpha=0.7)

    # Correlation
    corr = np.corrcoef(r_flat, py_flat)[0, 1]
    max_diff = np.max(np.abs(py_flat - r_flat))

    ax.set_title(f"{r['name']}\n{r['distance']} ({scale_label})", fontsize=9)
    ax.set_xlabel(f"R output ({scale_label})", fontsize=8)
    ax.set_ylabel(f"Python output ({scale_label})", fontsize=8)
    ax.text(
        0.05,
        0.95,
        f"r = {corr:.15f}\nmax|diff| = {max_diff:.2e}",
        transform=ax.transAxes,
        fontsize=7,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
    )

plt.suptitle("Python vs R: All Values", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Parity: Residual analysis (Python - R)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, r in enumerate(results):
    ax = axes[idx]

    # Compute residuals in log2 space for log2 datasets
    if r["use_log2"]:
        py_flat = np.log2(np.clip(r["py_output"].ravel(), 1e-300, None))
        r_flat = np.log2(np.clip(r["r_output"].ravel(), 1e-300, None))
        scale_label = "log2"
    else:
        py_flat = r["py_output"].ravel()
        r_flat = r["r_output"].ravel()
        scale_label = "linear"

    diff = py_flat - r_flat

    ax.hist(diff, bins=50, edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="r", linestyle="--", linewidth=0.5)

    ax.set_title(f"{r['name']} {r['distance']} ({scale_label})", fontsize=9)
    ax.set_xlabel(f"Python - R ({scale_label})", fontsize=8)
    ax.set_ylabel("Count", fontsize=8)
    ax.text(
        0.05,
        0.95,
        f"mean = {np.mean(diff):.2e}\nstd = {np.std(diff):.2e}\nmax = {np.max(np.abs(diff)):.2e}",
        transform=ax.transAxes,
        fontsize=7,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.5),
    )

plt.suptitle("Residuals: Python - R (should be ~0)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Parity: Only imputed values

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for idx, r in enumerate(results):
    ax = axes[idx]
    missing_mask = np.isnan(r["input"])

    # Compare imputed values in log2 space where appropriate
    if r["use_log2"]:
        py_imputed = np.log2(np.clip(r["py_output"][missing_mask], 1e-300, None))
        r_imputed = np.log2(np.clip(r["r_output"][missing_mask], 1e-300, None))
        scale_label = "log2"
    else:
        py_imputed = r["py_output"][missing_mask]
        r_imputed = r["r_output"][missing_mask]
        scale_label = "linear"

    ax.scatter(r_imputed, py_imputed, s=5, alpha=0.7, color="darkorange")

    lims = [
        min(r_imputed.min(), py_imputed.min()),
        max(r_imputed.max(), py_imputed.max()),
    ]
    ax.plot(lims, lims, "r--", linewidth=0.5)

    corr = np.corrcoef(r_imputed, py_imputed)[0, 1]
    max_abs_diff = np.max(np.abs(py_imputed - r_imputed))

    ax.set_title(
        f"{r['name']} {r['distance']}\n({len(py_imputed)} imputed, {scale_label})",
        fontsize=9,
    )
    ax.set_xlabel(f"R imputed ({scale_label})", fontsize=8)
    ax.set_ylabel(f"Python imputed ({scale_label})", fontsize=8)
    ax.text(
        0.05,
        0.95,
        f"r = {corr:.15f}\nmax|diff| = {max_abs_diff:.2e}",
        transform=ax.transAxes,
        fontsize=7,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.5),
    )

plt.suptitle("Python vs R: Imputed Values Only", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Performance: Python vs R timing

In [ ]:
# Build performance summary
perf_data = []
for r in results:
    if r["r_time"] is not None:
        perf_data.append(
            {
                "dataset": f"{r['name']}_{r['distance']}",
                "features": r["features"],
                "R (s)": r["r_time"],
                "Python (s)": r["py_time"],
                "Speedup": r["r_time"] / r["py_time"],
            }
        )

perf_df = pd.DataFrame(perf_data)
print(perf_df.to_string(index=False, float_format="{:.3f}".format))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: R vs Python times
x = np.arange(len(perf_df))
width = 0.35
ax1.barh(x - width / 2, perf_df["R (s)"], width, label="R", color="steelblue")
ax1.barh(
    x + width / 2,
    perf_df["Python (s)"],
    width,
    label="Python (Cython)",
    color="darkorange",
)
ax1.set_yticks(x)
ax1.set_yticklabels(perf_df["dataset"], fontsize=8)
ax1.set_xlabel("Time (seconds)")
ax1.set_title("Execution Time: R vs Python")
ax1.legend()
ax1.invert_yaxis()

# Speedup chart
colors = ["green" if s >= 1 else "red" for s in perf_df["Speedup"]]
ax2.barh(x, perf_df["Speedup"], color=colors, alpha=0.7)
ax2.axvline(1.0, color="black", linestyle="--", linewidth=0.5)
ax2.set_yticks(x)
ax2.set_yticklabels(perf_df["dataset"], fontsize=8)
ax2.set_xlabel("Speedup (Python/R, >1 = Python faster)")
ax2.set_title("Python Speedup over R")
ax2.invert_yaxis()

for i, s in enumerate(perf_df["Speedup"]):
    ax2.text(s + 0.1, i, f"{s:.1f}x", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Parity summary table

In [ ]:
summary = []
for r in results:
    diff = r["py_output"] - r["r_output"]
    missing_mask = np.isnan(r["input"])
    imputed_diff = diff[missing_mask]

    summary.append(
        {
            "dataset": f"{r['name']}_{r['distance']}",
            "features": r["features"],
            "samples": r["samples"],
            "n_missing": r["n_missing"],
            "max_abs_diff": np.max(np.abs(diff)),
            "max_imputed_diff": np.max(np.abs(imputed_diff)),
            "mean_imputed_diff": np.mean(np.abs(imputed_diff)),
            "parity_pass": np.allclose(
                r["py_output"], r["r_output"], rtol=1e-10, atol=1e-10
            ),
        }
    )

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print(f"\nAll parity tests pass: {summary_df['parity_pass'].all()}")